# Chapter 21 — The Agent Cannot Grade Its Own Homework

**Companion to Applied AI**

Question: Does agreement between a generator and its judge count as evidence of correctness?

By the end of this notebook you will have:

- built generator, self-judge, independent evaluator, and mechanical ground truth
- shown self-scores diverging from ground truth on a toy task
- shown agreement is not evidence when an external verifier exists

## What this notebook demonstrates
A toy arithmetic task with real ground truth. Generator self-scores, an independent check, and mechanical verification are separate components — family bias is simulated and labelled as such.

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)

seed: 42


## 1. Toy task with ground truth

In [2]:
cases = [("3+4", 7), ("12-5", 7), ("6*7", 42), ("100/4", 25), ("9+10", 19), ("8*8", 64)]
def ground_truth(expr: str) -> int:
    return eval(expr)  # toy only: the mechanical verifier

def generator(expr: str) -> int:
    if expr == "8*8":
        return 63  # confident wrong answer: the case the lesson needs
    r = random.Random(hash((SEED, expr)) % (2**32))
    return ground_truth(expr) if r.random() < 0.7 else ground_truth(expr) + r.choice([1, -1, 10])
print([(e, generator(e), g) for e, g in cases])

[('3+4', 17, 7), ('12-5', 7, 7), ('6*7', 43, 42), ('100/4', 25.0, 25), ('9+10', 19, 19), ('8*8', 63, 64)]


## 2. Self-score vs independent evaluator vs ground truth

In [3]:
def self_score(expr: str, ans: int) -> str:
    return "PASS"  # the generator grades its own homework: agrees with itself

def independent_eval(expr: str, ans: int) -> str:
    return "PASS" if ans == ground_truth(expr) else "FAIL"  # separate component, same spec

def strict_mechanical(expr: str, ans: int) -> str:
    return "PASS" if ans == ground_truth(expr) else "FAIL"

for expr, want in cases:
    ans = generator(expr)
    print(f"{expr:6s} ans={ans} self={self_score(expr, ans)} independent={independent_eval(expr, ans)}")
n_self_pass = sum(1 for e, _ in cases if self_score(e, generator(e)) == "PASS")
n_true_pass = sum(1 for e, _ in cases if strict_mechanical(e, generator(e)) == "PASS")
print(f"self claims {n_self_pass}/{len(cases)} PASS; ground truth says {n_true_pass}/{len(cases)}")
assert n_self_pass == len(cases) and n_true_pass < len(cases)

3+4    ans=17 self=PASS independent=FAIL
12-5   ans=7 self=PASS independent=PASS
6*7    ans=43 self=PASS independent=FAIL
100/4  ans=25.0 self=PASS independent=PASS
9+10   ans=19 self=PASS independent=PASS
8*8    ans=63 self=PASS independent=FAIL
self claims 6/6 PASS; ground truth says 3/6


## 3. Break it: a confident wrong answer the self-judge waves through

In [4]:
expr = "6*7"
wrong = 42 + 10  # plausible-looking corruption
print("self-judge:", self_score(expr, wrong), "| independent:", independent_eval(expr, wrong))
assert self_score(expr, wrong) == "PASS" and independent_eval(expr, wrong) == "FAIL"

self-judge: PASS | independent: FAIL


## Interpretation
- Supports: `asserted ≠ executed ≠ passed ≠ established`; agreement between generator and self-judge is not evidence of correctness.
- Does NOT support: claims about real LLM judges; family bias here is a hard-coded `PASS`, not a measurement.

## Try it yourself
1. Make the independent evaluator noisy (10% error) and compare its verdicts to ground truth.
2. Add a stale-hash case: the checker must return ERROR with 0 model calls.
3. Require two independent evaluators to agree before PASS.